# ARE-008A/B: training data scaling and fixed-validation model comparison
Real 1k / 5k / 10k, local saved artifacts only. No source downloads, test reads, training or tuning in this notebook. Full dataset quality audit was completed separately in ARE-008A; this notebook now uses the selective train/validation reader.

In [ ]:
from pathlib import Path
from IPython.display import display
from anime_recommender.training.scaling_experiments import (selective_scaling_overview, check_comparison_compatibility, load_comparison)
from anime_recommender.evaluation.scaling_validation import load_scaling_resources
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SCALING = ROOT / "data/processed/scaling"

## Compatibility, fixed validation and train-only vocabulary
The compatibility check verifies identical validation source fields, nested train users/items, preparation settings and baseline model fingerprints. It never reads test files.

In [ ]:
display(check_comparison_compatibility(ROOT))

## Real training data
Selected/clean/positive totals are metadata recorded by ARE-008A. Only train and validation interaction files are opened. Existing test artifacts are untouched.

In [ ]:
overview = selective_scaling_overview(ROOT)
display(overview["summary"])

## Train interaction distribution and validation coverage
Warm/cold context and target proportions use each model vocabulary. Validation source histories/targets remain fixed.

In [ ]:
display(overview["train_activity"])

## ??????? ??????????????? ???????
Windows PeakWorkingSet ???????? ??????????; ????????? ????????? ?????? ?? ?????? ???????? CSV.

In [ ]:
display(load_scaling_resources(SCALING))

## Scope and privacy
No YummyAnime requests or profile history are used in this comparison. Same rating threshold=5, seed=42 and train k-core=5/5. New models are trained only by the explicit offline ARE-008B command; notebook switching and charting never fit models.

## ARE-008B: model comparison on fixed validation
Loads saved comparison only: no training, test evaluation, tuning or franchise filtering. Full metrics include cold targets; Warm Recall is conditional on targets in each vocabulary. The approved default is 10k; statistical superiority over 5k is not established. Historical comparison metadata records the default at experiment time.

In [ ]:
from anime_recommender.training.scaling_experiments import load_comparison
comparison_result = load_comparison(ROOT)
display(comparison_result["comparison"])
display(comparison_result["metadata"]["recommended_model_version"])

In [ ]:
import matplotlib.pyplot as plt
quality = comparison_result["comparison"].set_index("model_version")
quality[["als_full_recall_10", "als_full_ndcg_10", "als_warm_recall_10"]].plot(marker="o", title="ALS validation quality vs training sample")
plt.ylabel("Metric @10")
plt.show()
quality[["popularity_recall_10", "popularity_ndcg_10"]].plot(marker="o", title="Popularity validation quality")
plt.show()
quality[["catalog_coverage_10", "cold_target_share"]].plot.bar(title="Catalog exposure and cold validation targets")
plt.show()

## Same validation user, three saved ALS versions
Targets are used only for is_hit annotation after ranking. No franchise filtering here. Higher scores across different models are not necessarily comparable.

In [ ]:
display(comparison_result["metadata"]["validation_user_id"])
display(comparison_result["validation_example"][["model_version", "rank", "title", "score", "is_hit", "myanimelist_id"]])